# Download Sept-Nov 2025 Harbin Sentinel-2 scenes (Colab)

Runs `download-haerbing-sepnov-sentinel2-data.py` from Colab instead of your local machine. The script itself lives in the repo and is unchanged -- this notebook just handles two things Colab needs that your local `venv` run didn't:

1. **Credentials**: `.env.cdse` is deliberately gitignored (a CDSE username/password should never be pushed to a public repo -- see the file's own setup comment), so it isn't in the cloned repo on Colab. Fix: upload your local `.env.cdse` to Google Drive once, next to your dataset tars, and this notebook loads it from there into the session's environment variables -- it is never written to the Colab VM's disk or printed anywhere in this notebook's output.
2. **Output location**: the script downloads scenes to `Haerbing_Dataset_sepnov/` on local disk. Colab's disk is wiped when the runtime disconnects, so this notebook copies the finished result to Drive at the end -- do this in the *same* runtime as the download (not a fresh session later).

**One-time setup before running this notebook**: in Google Drive, go to the same `COMP0173/` folder you uploaded the ground-truth tars to, and upload your local `.env.cdse` file there (Drive web UI: drag and drop, or right-click > File upload). It's a two-line text file, no need to tar it.

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = "/content/COMP0173_poster_pre"
    if not os.path.exists(REPO_DIR):
        !git clone -q https://github.com/jy-gfm/COMP0173_poster_pre.git {REPO_DIR}
    os.chdir(REPO_DIR)

    DRIVE_DIR = "/content/drive/MyDrive/COMP0173/"
else:
    DRIVE_DIR = "./"


## Load credentials from Drive (not from the repo -- `.env.cdse` is gitignored on purpose)

In [ ]:
def load_env_file(path):
    if not os.path.exists(path):
        raise RuntimeError(
            f"{path} not found. Upload your local .env.cdse to the COMP0173/ "
            "folder in Google Drive first (see this notebook's opening cell)."
        )
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()

load_env_file(os.path.join(DRIVE_DIR, ".env.cdse"))
assert os.environ.get("CDSE_USERNAME") and os.environ.get("CDSE_PASSWORD"), "credentials didn\'t load"
print("Credentials loaded into this session only -- not written to disk, not printed.")


## Run the download script

Same script as the local `venv` version -- `download-haerbing-sepnov-sentinel2-data.py` already reads `CDSE_USERNAME`/`CDSE_PASSWORD` from the environment, which the cell above just populated. This can take a while (14+ scenes, ~1GB each) -- if the runtime disconnects partway through, just re-run this cell: the script skips any `.SAFE` folder it already downloaded.

In [ ]:
!python download-haerbing-sepnov-sentinel2-data.py


## Manually check the downloaded scenes

Renders every downloaded scene as a small true-colour thumbnail in one grid, so you can visually scan for cloud cover, snow, or obviously bad scenes before spending time preprocessing all of them. Each thumbnail is a heavily decimated read (256x256, not the full 10980x10980) so this is fast -- it does not touch the full-resolution bands used later in `preprocess-haerbing-sepnov-sentinel2-data.py`.

In [ ]:
import os
import glob
import re
import numpy as np
import rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor

# Defined here (not just inside the download script) because this cell runs
# in the notebook's own Python session, separate from the `!python ...`
# subprocess that did the actual downloading -- it has no access to that
# subprocess's variables.
OUTPUT_DIR = "Haerbing_Dataset_sepnov"

SAFE_NAME_RE = re.compile(
    r"^(?P<sensor>S2[ABC])_MSIL2A_(?P<date>\d{8})T\d{6}_N\d{4}_R\d{3}_"
    r"(?P<tile>T\d{2}[A-Z]{3})_\d{8}T\d{6}\.SAFE$"
)
BOA_ADD_OFFSET = -1000  # same correction as preprocess-haerbing-sepnov-sentinel2-data.py
REFLECTANCE_SCALE = 10000

def find_granule_dir(safe_path):
    granule_root = f"{safe_path}/GRANULE"
    subdirs = [d for d in os.listdir(granule_root) if not d.startswith('.')]
    return f"{granule_root}/{subdirs[0]}"

def band_path(granule_dir, band):
    # Prefer the 60m band files that Sentinel-2 L2A already ships (native
    # ~1830x1830px) over decimating the 10980x10980px 10m files down to a
    # thumbnail -- decoding a full 10m JP2 just to average it away is the
    # actual bottleneck here (this is only a visual cloud/snow/artifact
    # check, not the training data, so 60m resolution is plenty).
    matches = glob.glob(f"{granule_dir}/IMG_DATA/R60m/*_{band}_60m.jp2")
    if not matches:
        matches = glob.glob(f"{granule_dir}/IMG_DATA/R10m/*_{band}_10m.jp2")
    return matches[0]

def quicklook_rgb(safe_path, size=256):
    granule_dir = find_granule_dir(safe_path)
    def read_small(band):
        with rasterio.open(band_path(granule_dir, band)) as src:
            arr = src.read(1, out_shape=(size, size), resampling=Resampling.average).astype(np.float32)
        return np.clip((arr + BOA_ADD_OFFSET) / REFLECTANCE_SCALE, 0.0, 1.0)
    rgb = np.stack([read_small(b) for b in ("B04", "B03", "B02")], axis=-1)
    stretched = np.zeros_like(rgb)
    for c in range(3):
        lo, hi = np.percentile(rgb[:, :, c], [2, 98])
        stretched[:, :, c] = np.clip((rgb[:, :, c] - lo) / max(hi - lo, 1e-6), 0, 1)
    return np.clip(stretched, 0, 1) ** 0.6

def safe_quicklook(safe_path):
    try:
        return quicklook_rgb(safe_path)
    except Exception as e:
        return e

safe_dirs = sorted(glob.glob(f"{OUTPUT_DIR}/*.SAFE"))
print(f"{len(safe_dirs)} downloaded scenes to preview")

# Reading is I/O + JP2-decode bound and each scene is independent, so fan
# these out across threads (rasterio/GDAL releases the GIL during decode) --
# this is the other big chunk of the wall-clock time on 70+ scenes.
with ThreadPoolExecutor(max_workers=8) as pool:
    thumbnails = list(pool.map(safe_quicklook, safe_dirs))

cols = 8
rows = (len(safe_dirs) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2.3))
axes = axes.flatten()
for i, (safe_path, thumb) in enumerate(zip(safe_dirs, thumbnails)):
    m = SAFE_NAME_RE.match(os.path.basename(safe_path))
    if isinstance(thumb, Exception):
        axes[i].text(0.5, 0.5, f"error:\n{thumb}", ha="center", va="center", fontsize=6, wrap=True)
    else:
        axes[i].imshow(thumb)
    axes[i].set_title(f"[{i}] {m.group('tile')} {m.group('date')}", fontsize=8)
    axes[i].axis("off")
for j in range(len(safe_dirs), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

## Exclude any bad scenes you spotted above

Look at the grid above and note the `[index]` of any scene you'd rather not use (heavy cloud, snow, obvious artefacts). List those indices below -- this **moves** (doesn't delete) those `.SAFE` folders out to a separate `_excluded` folder, so they're reversible and won't be included in the tar/Drive upload or in preprocessing. Leave the list empty to keep everything.

In [ ]:
import shutil

EXCLUDE_INDICES = []  # e.g. [3, 17, 42] -- fill in after looking at the grid above

EXCLUDED_DIR = f"{OUTPUT_DIR}_excluded"
os.makedirs(EXCLUDED_DIR, exist_ok=True)

for i in EXCLUDE_INDICES:
    safe_path = safe_dirs[i]
    dest = os.path.join(EXCLUDED_DIR, os.path.basename(safe_path))
    shutil.move(safe_path, dest)
    print(f"excluded [{i}]: {os.path.basename(safe_path)}")

print(f"\n{len(EXCLUDE_INDICES)} scene(s) moved to {EXCLUDED_DIR}, "
      f"{len(safe_dirs) - len(EXCLUDE_INDICES)} remain in {OUTPUT_DIR}")


## Copy the result to Drive

Do this before the runtime disconnects -- `Haerbing_Dataset_sepnov/` on Colab's local disk does not survive a disconnect. A single tar archive is more reliable to upload/download than thousands of individual files (same reasoning as every other dataset in this project).

In [ ]:
if IN_COLAB:
    TAR_NAME = "Haerbing_Dataset_sepnov"
    !tar -cf {TAR_NAME}.tar {TAR_NAME}
    import shutil
    shutil.copy(f"{TAR_NAME}.tar", DRIVE_DIR)
    print(f"Saved {TAR_NAME}.tar to {DRIVE_DIR}")
